# Hierarchical Clustering - From Scratch Implementation

## Table of Contents
1. [Theory & Mathematical Foundation](#theory)
2. [Implementation from Scratch](#implementation)
3. [Training & Optimization](#training)
4. [Diagnostics & Evaluation](#diagnostics)
5. [Visualizations](#visualizations)
6. [Use Cases & Guidelines](#use-cases)
7. [Comparison with sklearn](#comparison)

In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage as scipy_linkage, cophenet, fcluster
from scipy.spatial.distance import pdist, squareform
from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.cluster import AgglomerativeClustering as SklearnAgglomerative
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## 1. Theory & Mathematical Foundation <a id='theory'></a>

### What is Hierarchical Clustering?

Hierarchical clustering is an unsupervised learning algorithm that builds a **hierarchy of clusters**. Unlike K-Means, it doesn't require specifying the number of clusters beforehand and provides a tree-like structure (dendrogram) showing relationships between data points.

### Two Approaches

#### 1. Agglomerative (Bottom-Up)
- **Start**: Each point is its own cluster
- **Process**: Iteratively merge the two closest clusters
- **End**: All points in one cluster
- **Most common approach**

#### 2. Divisive (Top-Down)
- **Start**: All points in one cluster
- **Process**: Recursively split clusters
- **End**: Each point is its own cluster
- **Less common, computationally expensive**

### Linkage Methods

The key decision in hierarchical clustering is how to measure the distance between clusters:

#### 1. Single Linkage (Minimum)
$$d(A, B) = \min_{a \in A, b \in B} d(a, b)$$
- Distance = minimum distance between any two points
- **Pros**: Can find elongated, non-spherical clusters
- **Cons**: Susceptible to "chaining" effect

#### 2. Complete Linkage (Maximum)
$$d(A, B) = \max_{a \in A, b \in B} d(a, b)$$
- Distance = maximum distance between any two points
- **Pros**: Produces compact, spherical clusters
- **Cons**: Sensitive to outliers

#### 3. Average Linkage (UPGMA)
$$d(A, B) = \frac{1}{|A| \cdot |B|} \sum_{a \in A} \sum_{b \in B} d(a, b)$$
- Distance = average of all pairwise distances
- **Pros**: Balanced approach, robust to outliers
- **Cons**: Computationally more expensive

#### 4. Ward's Method
$$d(A, B) = \sqrt{\frac{2|A||B|}{|A|+|B|}} ||\bar{a} - \bar{b}||_2$$
- Minimizes within-cluster variance when merging
- **Pros**: Produces compact, similarly-sized clusters
- **Cons**: Assumes spherical clusters

### Dendrogram Interpretation

A dendrogram visualizes the hierarchical structure:
- **Y-axis**: Distance/dissimilarity at which clusters merge
- **X-axis**: Individual data points or clusters
- **Horizontal lines**: Cluster merges
- **Cutting the dendrogram**: Draw a horizontal line to get desired number of clusters

### Time and Space Complexity

| Operation | Time Complexity | Space Complexity |
|-----------|----------------|------------------|
| Naive Agglomerative | O(n^3) | O(n^2) |
| Optimized (Single Linkage) | O(n^2) | O(n^2) |
| Distance Matrix | O(n^2 * d) | O(n^2) |

Where n = number of samples, d = number of features

**Important**: Due to O(n^2) memory for storing distance matrices, hierarchical clustering is not suitable for large datasets (typically n < 10,000).

## 2. Implementation from Scratch <a id='implementation'></a>

In [ ]:
class AgglomerativeClustering:
    """
    Agglomerative Hierarchical Clustering implementation from scratch.
    
    Parameters:
    -----------
    n_clusters : int, default=2
        The number of clusters to find.
    linkage : str, default='ward'
        Linkage criterion: 'single', 'complete', 'average', 'ward'
    
    Attributes:
    -----------
    labels_ : ndarray of shape (n_samples,)
        Cluster labels for each point.
    linkage_matrix_ : ndarray
        The hierarchical clustering encoded as a linkage matrix.
        Format: [cluster1, cluster2, distance, n_samples_in_new_cluster]
    n_features_ : int
        Number of features in the input data.
    """
    
    def __init__(self, n_clusters=2, linkage='ward'):
        self.n_clusters = n_clusters
        self.linkage = linkage
        self.labels_ = None
        self.linkage_matrix_ = None
        self.n_features_ = None
        
    def _compute_distance_matrix(self, X):
        """
        Compute pairwise Euclidean distance matrix.
        
        Parameters:
        -----------
        X : ndarray of shape (n_samples, n_features)
            Input data.
            
        Returns:
        --------
        distance_matrix : ndarray of shape (n_samples, n_samples)
            Pairwise distance matrix.
        """
        n_samples = X.shape[0]
        distance_matrix = np.zeros((n_samples, n_samples))
        
        for i in range(n_samples):
            for j in range(i + 1, n_samples):
                dist = np.sqrt(np.sum((X[i] - X[j]) ** 2))
                distance_matrix[i, j] = dist
                distance_matrix[j, i] = dist
                
        return distance_matrix
    
    def _single_linkage(self, dist_matrix, cluster_i, cluster_j):
        """
        Single linkage: minimum distance between clusters.
        """
        return np.min(dist_matrix[np.ix_(cluster_i, cluster_j)])
    
    def _complete_linkage(self, dist_matrix, cluster_i, cluster_j):
        """
        Complete linkage: maximum distance between clusters.
        """
        return np.max(dist_matrix[np.ix_(cluster_i, cluster_j)])
    
    def _average_linkage(self, dist_matrix, cluster_i, cluster_j):
        """
        Average linkage: mean distance between clusters.
        """
        return np.mean(dist_matrix[np.ix_(cluster_i, cluster_j)])
    
    def _ward_linkage(self, X, cluster_i, cluster_j):
        """
        Ward's method: minimize within-cluster variance.
        
        The distance is computed as:
        d(A, B) = sqrt(2 * |A| * |B| / (|A| + |B|)) * ||centroid_A - centroid_B||_2
        """
        centroid_i = np.mean(X[cluster_i], axis=0)
        centroid_j = np.mean(X[cluster_j], axis=0)
        n_i = len(cluster_i)
        n_j = len(cluster_j)
        
        # Ward's distance formula
        dist = np.sqrt(2 * n_i * n_j / (n_i + n_j)) * np.linalg.norm(centroid_i - centroid_j)
        return dist
    
    def _get_cluster_distance(self, X, dist_matrix, cluster_i, cluster_j):
        """
        Compute distance between two clusters based on linkage method.
        """
        if self.linkage == 'single':
            return self._single_linkage(dist_matrix, cluster_i, cluster_j)
        elif self.linkage == 'complete':
            return self._complete_linkage(dist_matrix, cluster_i, cluster_j)
        elif self.linkage == 'average':
            return self._average_linkage(dist_matrix, cluster_i, cluster_j)
        elif self.linkage == 'ward':
            return self._ward_linkage(X, cluster_i, cluster_j)
        else:
            raise ValueError(f"Unknown linkage method: {self.linkage}")
    
    def fit(self, X):
        """
        Fit the hierarchical clustering model.
        
        Parameters:
        -----------
        X : array-like of shape (n_samples, n_features)
            Training data.
            
        Returns:
        --------
        self : object
            Fitted estimator.
        """
        X = np.array(X)
        n_samples, self.n_features_ = X.shape
        
        # Compute initial distance matrix
        dist_matrix = self._compute_distance_matrix(X)
        
        # Initialize clusters - each point is its own cluster
        # clusters[i] contains list of original point indices in cluster i
        clusters = {i: [i] for i in range(n_samples)}
        
        # Linkage matrix to store merge history
        # Format: [cluster1_id, cluster2_id, distance, n_points_in_merged_cluster]
        self.linkage_matrix_ = []
        
        # Next cluster id (for newly formed clusters)
        next_cluster_id = n_samples
        
        # Agglomerative clustering: merge until we have desired number of clusters
        while len(clusters) > 1:
            # Find the two closest clusters
            min_dist = np.inf
            merge_i, merge_j = None, None
            
            cluster_ids = list(clusters.keys())
            
            for i in range(len(cluster_ids)):
                for j in range(i + 1, len(cluster_ids)):
                    id_i, id_j = cluster_ids[i], cluster_ids[j]
                    dist = self._get_cluster_distance(
                        X, dist_matrix, clusters[id_i], clusters[id_j]
                    )
                    
                    if dist < min_dist:
                        min_dist = dist
                        merge_i, merge_j = id_i, id_j
            
            # Merge the two closest clusters
            new_cluster = clusters[merge_i] + clusters[merge_j]
            n_points = len(new_cluster)
            
            # Record the merge in linkage matrix
            self.linkage_matrix_.append([merge_i, merge_j, min_dist, n_points])
            
            # Remove old clusters and add new merged cluster
            del clusters[merge_i]
            del clusters[merge_j]
            clusters[next_cluster_id] = new_cluster
            next_cluster_id += 1
        
        # Convert linkage matrix to numpy array
        self.linkage_matrix_ = np.array(self.linkage_matrix_)
        
        # Assign cluster labels based on n_clusters
        self._assign_labels(n_samples)
        
        return self
    
    def _assign_labels(self, n_samples):
        """
        Assign cluster labels by cutting the dendrogram at the appropriate level.
        """
        # Start with each point in its own cluster
        # clusters maps cluster_id -> list of original point indices
        clusters = {i: [i] for i in range(n_samples)}
        next_id = n_samples
        
        # Number of merges to perform = n_samples - n_clusters
        n_merges = n_samples - self.n_clusters
        
        # Replay merges up to the desired level
        for i in range(n_merges):
            merge_i = int(self.linkage_matrix_[i, 0])
            merge_j = int(self.linkage_matrix_[i, 1])
            
            # Merge clusters
            new_cluster = clusters[merge_i] + clusters[merge_j]
            del clusters[merge_i]
            del clusters[merge_j]
            clusters[next_id] = new_cluster
            next_id += 1
        
        # Assign labels
        self.labels_ = np.zeros(n_samples, dtype=int)
        for label, (cluster_id, points) in enumerate(clusters.items()):
            for point_idx in points:
                self.labels_[point_idx] = label
    
    def fit_predict(self, X):
        """
        Fit the model and return cluster labels.
        
        Parameters:
        -----------
        X : array-like of shape (n_samples, n_features)
            Training data.
            
        Returns:
        --------
        labels : ndarray of shape (n_samples,)
            Cluster labels.
        """
        self.fit(X)
        return self.labels_
    
    def get_dendrogram_data(self):
        """
        Get the linkage matrix for plotting dendrogram.
        
        Returns:
        --------
        linkage_matrix : ndarray
            Linkage matrix compatible with scipy's dendrogram function.
        """
        return self.linkage_matrix_

In [ ]:
# Quick test of our implementation
print("Testing AgglomerativeClustering implementation...")
print("=" * 50)

# Create simple test data
X_test = np.array([[1, 2], [1.5, 1.8], [5, 8], [8, 8], [1, 0.6], [9, 11]])

# Test with different linkage methods
for linkage in ['single', 'complete', 'average', 'ward']:
    model = AgglomerativeClustering(n_clusters=2, linkage=linkage)
    labels = model.fit_predict(X_test)
    print(f"\n{linkage.capitalize()} Linkage:")
    print(f"  Labels: {labels}")
    print(f"  Linkage Matrix Shape: {model.linkage_matrix_.shape}")

## 3. Training & Optimization <a id='training'></a>

In [ ]:
# Generate synthetic datasets with different characteristics
# Using small datasets due to O(n^2) complexity

# Dataset 1: Well-separated blobs
X_blobs, y_blobs = make_blobs(
    n_samples=150, 
    n_features=2, 
    centers=3, 
    cluster_std=0.8, 
    random_state=42
)

# Dataset 2: Varied cluster sizes and densities
X_varied, y_varied = make_blobs(
    n_samples=[50, 80, 100],
    n_features=2,
    centers=[(-5, -5), (0, 0), (5, 5)],
    cluster_std=[0.5, 1.5, 0.8],
    random_state=42
)

# Dataset 3: Elongated clusters (good for single linkage)
X_moons, y_moons = make_moons(n_samples=150, noise=0.1, random_state=42)

# Dataset 4: Concentric circles
X_circles, y_circles = make_circles(n_samples=150, noise=0.05, factor=0.5, random_state=42)

# Standardize all datasets
scaler = StandardScaler()
datasets = {
    'Blobs': (scaler.fit_transform(X_blobs), y_blobs, 3),
    'Varied': (scaler.fit_transform(X_varied), y_varied, 3),
    'Moons': (scaler.fit_transform(X_moons), y_moons, 2),
    'Circles': (scaler.fit_transform(X_circles), y_circles, 2)
}

print("Datasets created:")
for name, (X, y, n_clusters) in datasets.items():
    print(f"  {name}: {X.shape[0]} samples, {n_clusters} clusters")

In [ ]:
# Visualize the datasets
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for idx, (name, (X, y, _)) in enumerate(datasets.items()):
    axes[idx].scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', s=30, alpha=0.7)
    axes[idx].set_title(f'{name} Dataset (True Labels)')
    axes[idx].set_xlabel('Feature 1')
    axes[idx].set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

In [ ]:
# Train our implementation on different datasets and linkage methods
linkage_methods = ['single', 'complete', 'average', 'ward']
results = {}

print("Training Hierarchical Clustering Models")
print("=" * 60)

for dataset_name, (X, y_true, n_clusters) in datasets.items():
    results[dataset_name] = {}
    print(f"\n{dataset_name} Dataset:")
    
    for linkage in linkage_methods:
        # Fit our model
        model = AgglomerativeClustering(n_clusters=n_clusters, linkage=linkage)
        labels = model.fit_predict(X)
        
        # Calculate metrics
        sil_score = silhouette_score(X, labels) if len(np.unique(labels)) > 1 else 0
        ari_score = adjusted_rand_score(y_true, labels)
        
        results[dataset_name][linkage] = {
            'model': model,
            'labels': labels,
            'silhouette': sil_score,
            'ari': ari_score
        }
        
        print(f"  {linkage:10s}: Silhouette={sil_score:.3f}, ARI={ari_score:.3f}")

## 4. Diagnostics & Evaluation <a id='diagnostics'></a>

In [ ]:
def compute_silhouette_scores(X, max_clusters=10):
    """
    Compute silhouette scores for different numbers of clusters.
    Helps determine optimal number of clusters.
    """
    silhouette_scores = []
    cluster_range = range(2, min(max_clusters + 1, len(X)))
    
    for n_clusters in cluster_range:
        model = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
        labels = model.fit_predict(X)
        score = silhouette_score(X, labels)
        silhouette_scores.append(score)
    
    return list(cluster_range), silhouette_scores


def compute_cophenetic_correlation(X, linkage_method='ward'):
    """
    Compute cophenetic correlation coefficient.
    
    Measures how faithfully the dendrogram preserves pairwise distances.
    Values closer to 1 indicate better preservation.
    """
    # Use scipy for accurate cophenetic calculation
    linkage_matrix = scipy_linkage(X, method=linkage_method)
    coph_corr, coph_dists = cophenet(linkage_matrix, pdist(X))
    return coph_corr

In [ ]:
# Silhouette analysis for optimal cluster selection
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, (name, (X, y_true, true_n)) in enumerate(datasets.items()):
    cluster_range, sil_scores = compute_silhouette_scores(X, max_clusters=8)
    
    axes[idx].plot(cluster_range, sil_scores, 'bo-', linewidth=2, markersize=8)
    axes[idx].axvline(x=true_n, color='red', linestyle='--', 
                      label=f'True clusters: {true_n}')
    
    # Mark optimal
    optimal_k = cluster_range[np.argmax(sil_scores)]
    axes[idx].axvline(x=optimal_k, color='green', linestyle=':', 
                      label=f'Optimal (silhouette): {optimal_k}')
    
    axes[idx].set_xlabel('Number of Clusters')
    axes[idx].set_ylabel('Silhouette Score')
    axes[idx].set_title(f'{name} - Silhouette Analysis')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Cophenetic correlation for different linkage methods
print("Cophenetic Correlation Coefficients")
print("=" * 60)
print("\nHigher values indicate better preservation of original distances.")
print("\n" + "-" * 60)
print(f"{'Dataset':<15} {'Single':>10} {'Complete':>10} {'Average':>10} {'Ward':>10}")
print("-" * 60)

for name, (X, _, _) in datasets.items():
    coph_scores = []
    for linkage in ['single', 'complete', 'average', 'ward']:
        coph = compute_cophenetic_correlation(X, linkage)
        coph_scores.append(coph)
    
    print(f"{name:<15} {coph_scores[0]:>10.3f} {coph_scores[1]:>10.3f} "
          f"{coph_scores[2]:>10.3f} {coph_scores[3]:>10.3f}")

print("-" * 60)

In [ ]:
# Comprehensive evaluation table
print("\nComprehensive Evaluation Summary")
print("=" * 80)

for dataset_name in datasets.keys():
    print(f"\n{dataset_name} Dataset:")
    print("-" * 60)
    print(f"{'Linkage':<12} {'Silhouette':>12} {'ARI':>12} {'Unique Clusters':>18}")
    print("-" * 60)
    
    for linkage in linkage_methods:
        res = results[dataset_name][linkage]
        n_unique = len(np.unique(res['labels']))
        print(f"{linkage:<12} {res['silhouette']:>12.3f} {res['ari']:>12.3f} {n_unique:>18}")

## 5. Visualizations <a id='visualizations'></a>

In [ ]:
def plot_dendrogram(X, linkage_method='ward', title='Dendrogram', ax=None, 
                    truncate_mode=None, p=30, color_threshold=None):
    """
    Plot dendrogram using scipy.
    
    Parameters:
    -----------
    X : array-like
        Data matrix.
    linkage_method : str
        Linkage method for clustering.
    truncate_mode : str or None
        Truncation mode ('lastp', 'level', or None).
    p : int
        For truncation - number of leaves or levels to show.
    color_threshold : float or None
        Threshold for coloring clusters.
    """
    # Compute linkage matrix using scipy
    Z = scipy_linkage(X, method=linkage_method)
    
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 6))
    
    # Plot dendrogram
    dendrogram(
        Z,
        ax=ax,
        truncate_mode=truncate_mode,
        p=p,
        color_threshold=color_threshold,
        leaf_rotation=90,
        leaf_font_size=8
    )
    
    ax.set_title(title)
    ax.set_xlabel('Sample Index')
    ax.set_ylabel('Distance')
    
    return Z

In [ ]:
# Plot dendrograms for different linkage methods on Blobs dataset
X_demo, y_demo, n_demo = datasets['Blobs']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for idx, linkage in enumerate(linkage_methods):
    ax = axes[idx // 2, idx % 2]
    plot_dendrogram(
        X_demo, 
        linkage_method=linkage, 
        title=f'{linkage.capitalize()} Linkage Dendrogram',
        ax=ax
    )

plt.tight_layout()
plt.show()

In [ ]:
# Dendrogram with cutting threshold visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Compute linkage
Z = scipy_linkage(X_demo, method='ward')

# Plot with different color thresholds (cutting points)
for idx, n_clusters in enumerate([2, 3]):
    # Find threshold that gives desired number of clusters
    # Threshold is the distance at which we cut
    threshold = Z[-(n_clusters-1), 2] if n_clusters > 1 else Z[-1, 2] + 1
    
    dendrogram(
        Z,
        ax=axes[idx],
        color_threshold=threshold,
        leaf_rotation=90,
        leaf_font_size=8
    )
    
    # Draw cutting line
    axes[idx].axhline(y=threshold, color='red', linestyle='--', linewidth=2,
                      label=f'Cut at {threshold:.2f}')
    axes[idx].set_title(f'Dendrogram Cut for {n_clusters} Clusters')
    axes[idx].set_xlabel('Sample Index')
    axes[idx].set_ylabel('Distance')
    axes[idx].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Compare cluster assignments across linkage methods
def plot_cluster_comparison(X, y_true, n_clusters, dataset_name):
    """
    Compare clustering results from different linkage methods.
    """
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    
    # True labels
    axes[0].scatter(X[:, 0], X[:, 1], c=y_true, cmap='viridis', s=30, alpha=0.7)
    axes[0].set_title('True Labels')
    axes[0].set_xlabel('Feature 1')
    axes[0].set_ylabel('Feature 2')
    
    # Different linkage methods
    for idx, linkage in enumerate(linkage_methods):
        model = AgglomerativeClustering(n_clusters=n_clusters, linkage=linkage)
        labels = model.fit_predict(X)
        
        sil = silhouette_score(X, labels) if len(np.unique(labels)) > 1 else 0
        
        axes[idx + 1].scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis', s=30, alpha=0.7)
        axes[idx + 1].set_title(f'{linkage.capitalize()}\nSilhouette: {sil:.3f}')
        axes[idx + 1].set_xlabel('Feature 1')
    
    plt.suptitle(f'{dataset_name} Dataset - Linkage Method Comparison', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

# Plot comparisons for each dataset
for name, (X, y, n_clusters) in datasets.items():
    plot_cluster_comparison(X, y, n_clusters, name)

In [ ]:
# Silhouette plot for cluster quality visualization
def plot_silhouette_diagram(X, labels, ax=None):
    """
    Plot silhouette diagram to visualize cluster quality.
    """
    from sklearn.metrics import silhouette_samples
    
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))
    
    n_clusters = len(np.unique(labels))
    silhouette_vals = silhouette_samples(X, labels)
    
    y_lower = 10
    colors = plt.cm.viridis(np.linspace(0, 1, n_clusters))
    
    for i in range(n_clusters):
        cluster_silhouette_vals = silhouette_vals[labels == i]
        cluster_silhouette_vals.sort()
        
        size_cluster_i = len(cluster_silhouette_vals)
        y_upper = y_lower + size_cluster_i
        
        ax.fill_betweenx(
            np.arange(y_lower, y_upper),
            0,
            cluster_silhouette_vals,
            facecolor=colors[i],
            edgecolor=colors[i],
            alpha=0.7
        )
        
        ax.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))
        y_lower = y_upper + 10
    
    avg_silhouette = np.mean(silhouette_vals)
    ax.axvline(x=avg_silhouette, color='red', linestyle='--', 
               label=f'Average: {avg_silhouette:.3f}')
    
    ax.set_xlabel('Silhouette Coefficient')
    ax.set_ylabel('Cluster')
    ax.set_xlim([-0.1, 1])
    ax.legend(loc='upper right')
    
    return avg_silhouette

# Plot silhouette diagrams for Blobs dataset with different linkage methods
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

X_sil, y_sil, n_sil = datasets['Blobs']

for idx, linkage in enumerate(linkage_methods):
    ax = axes[idx // 2, idx % 2]
    model = AgglomerativeClustering(n_clusters=n_sil, linkage=linkage)
    labels = model.fit_predict(X_sil)
    plot_silhouette_diagram(X_sil, labels, ax)
    ax.set_title(f'{linkage.capitalize()} Linkage - Silhouette Diagram')

plt.tight_layout()
plt.show()

## 6. Use Cases & Guidelines <a id='use-cases'></a>

### When to Use Hierarchical Clustering

#### Ideal Use Cases:

1. **Unknown Number of Clusters**
   - Dendrogram helps explore different cluster counts
   - No need to specify K beforehand
   - Can cut at different levels for different granularities

2. **Need Hierarchical Structure**
   - Taxonomy creation (biological, product categories)
   - Organizational structures
   - Document/topic hierarchies

3. **Exploratory Data Analysis**
   - Understanding data structure
   - Identifying natural groupings
   - Feature engineering insights

4. **Small to Medium Datasets**
   - n < 10,000 samples (practical limit)
   - Memory for O(n^2) distance matrix

5. **Non-Spherical Clusters** (with single linkage)
   - Elongated cluster shapes
   - Arbitrarily shaped clusters

#### Example Applications:
- **Biology**: Phylogenetic trees, gene expression analysis
- **Marketing**: Customer segmentation with hierarchy
- **NLP**: Document clustering, topic hierarchies
- **Image Analysis**: Image segmentation, object grouping

### When NOT to Use Hierarchical Clustering

#### Avoid When:

1. **Large Datasets (n > 10,000)**
   - O(n^2) or O(n^3) time complexity
   - O(n^2) memory for distance matrix
   - Use: K-Means, DBSCAN, Mini-batch K-Means

2. **Real-time Clustering Needed**
   - No incremental updates possible
   - Must recompute from scratch
   - Use: Online K-Means, streaming algorithms

3. **Spherical, Equal-sized Clusters Needed**
   - Ward's method assumes spherical clusters
   - K-Means often works better and faster

4. **High-Dimensional Data**
   - Distance metrics become less meaningful
   - Consider dimensionality reduction first

### Linkage Selection Guide

| Linkage | Best For | Avoid When | Cluster Shape |
|---------|----------|------------|---------------|
| **Single** | Elongated clusters, chain-like | Noisy data, outliers | Arbitrary |
| **Complete** | Compact clusters, outlier-robust | Varied cluster sizes | Spherical |
| **Average** | General purpose, balanced | Need interpretability | Moderate |
| **Ward** | Equal-sized, compact clusters | Non-spherical shapes | Spherical |

### Decision Flowchart

```
Start
  |
  v
n > 10,000? --Yes--> Use K-Means, DBSCAN, or MiniBatch K-Means
  |
  No
  |
  v
Need hierarchy? --No--> Consider K-Means (faster)
  |
  Yes
  |
  v
Know K? --Yes--> Still can use HC with cut
  |
  No
  |
  v
Cluster shape?
  |
  +-- Spherical --> Ward's Method
  +-- Elongated --> Single Linkage
  +-- Unknown --> Start with Average, try others
```

### Hyperparameter Tuning

| Parameter | Options | Selection Criteria |
|-----------|---------|--------------------|
| **Linkage** | single, complete, average, ward | Based on expected cluster shape |
| **Distance** | euclidean, manhattan, cosine | Based on data type |
| **n_clusters** | 2 to sqrt(n) | Dendrogram + silhouette analysis |

### Common Pitfalls

1. **Chaining Effect (Single Linkage)**
   - Problem: Points chain together inappropriately
   - Solution: Use complete or average linkage

2. **Memory Errors**
   - Problem: O(n^2) distance matrix too large
   - Solution: Sample data or use approximate methods

3. **Scale Sensitivity**
   - Problem: Features with large ranges dominate
   - Solution: Always standardize features

4. **Incorrect Cut Level**
   - Problem: Wrong number of clusters
   - Solution: Use silhouette analysis, domain knowledge

In [ ]:
# Demonstrate the chaining effect with single linkage
# Create data that shows the difference
np.random.seed(42)

# Two clusters connected by a few bridging points
cluster1 = np.random.randn(40, 2) + np.array([-3, 0])
cluster2 = np.random.randn(40, 2) + np.array([3, 0])
# Bridge points
bridge = np.array([[-1, 0], [0, 0], [1, 0]])
X_chain = np.vstack([cluster1, bridge, cluster2])

# Compare single vs complete linkage
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Original data
axes[0].scatter(X_chain[:, 0], X_chain[:, 1], c='blue', s=30, alpha=0.7)
axes[0].scatter(bridge[:, 0], bridge[:, 1], c='red', s=100, marker='x', 
                linewidths=3, label='Bridge points')
axes[0].set_title('Original Data with Bridge Points')
axes[0].legend()

# Single linkage - will chain
model_single = AgglomerativeClustering(n_clusters=2, linkage='single')
labels_single = model_single.fit_predict(X_chain)
axes[1].scatter(X_chain[:, 0], X_chain[:, 1], c=labels_single, cmap='viridis', s=30)
axes[1].set_title(f'Single Linkage\n(Chaining Effect)')

# Complete linkage - will separate
model_complete = AgglomerativeClustering(n_clusters=2, linkage='complete')
labels_complete = model_complete.fit_predict(X_chain)
axes[2].scatter(X_chain[:, 0], X_chain[:, 1], c=labels_complete, cmap='viridis', s=30)
axes[2].set_title(f'Complete Linkage\n(Separates Clusters)')

for ax in axes:
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

print("\nSingle linkage connected all points due to the bridge.")
print("Complete linkage correctly identified the two separate clusters.")

## 7. Comparison with sklearn <a id='comparison'></a>

In [ ]:
# Compare our implementation with sklearn
from sklearn.cluster import AgglomerativeClustering as SklearnAgglomerative
import time

print("Comparison: Our Implementation vs sklearn")
print("=" * 70)

# Use Blobs dataset for comparison
X_compare, y_compare, n_compare = datasets['Blobs']

comparison_results = []

for linkage in linkage_methods:
    # Our implementation
    start_time = time.time()
    our_model = AgglomerativeClustering(n_clusters=n_compare, linkage=linkage)
    our_labels = our_model.fit_predict(X_compare)
    our_time = time.time() - start_time
    our_silhouette = silhouette_score(X_compare, our_labels)
    our_ari = adjusted_rand_score(y_compare, our_labels)
    
    # sklearn implementation
    start_time = time.time()
    sklearn_model = SklearnAgglomerative(n_clusters=n_compare, linkage=linkage)
    sklearn_labels = sklearn_model.fit_predict(X_compare)
    sklearn_time = time.time() - start_time
    sklearn_silhouette = silhouette_score(X_compare, sklearn_labels)
    sklearn_ari = adjusted_rand_score(y_compare, sklearn_labels)
    
    # Label agreement (accounting for label permutation)
    label_agreement = adjusted_rand_score(our_labels, sklearn_labels)
    
    comparison_results.append({
        'linkage': linkage,
        'our_silhouette': our_silhouette,
        'sklearn_silhouette': sklearn_silhouette,
        'our_ari': our_ari,
        'sklearn_ari': sklearn_ari,
        'our_time': our_time,
        'sklearn_time': sklearn_time,
        'agreement': label_agreement
    })

# Print comparison table
print(f"\n{'Linkage':<12} {'Our Sil':>10} {'SK Sil':>10} {'Our ARI':>10} {'SK ARI':>10} {'Agreement':>12}")
print("-" * 70)

for res in comparison_results:
    print(f"{res['linkage']:<12} {res['our_silhouette']:>10.3f} {res['sklearn_silhouette']:>10.3f} "
          f"{res['our_ari']:>10.3f} {res['sklearn_ari']:>10.3f} {res['agreement']:>12.3f}")

In [ ]:
# Visual comparison
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for idx, linkage in enumerate(linkage_methods):
    # Our implementation
    our_model = AgglomerativeClustering(n_clusters=n_compare, linkage=linkage)
    our_labels = our_model.fit_predict(X_compare)
    
    # sklearn
    sklearn_model = SklearnAgglomerative(n_clusters=n_compare, linkage=linkage)
    sklearn_labels = sklearn_model.fit_predict(X_compare)
    
    # Plot our implementation
    axes[0, idx].scatter(X_compare[:, 0], X_compare[:, 1], c=our_labels, 
                         cmap='viridis', s=30, alpha=0.7)
    axes[0, idx].set_title(f'Our - {linkage.capitalize()}')
    
    # Plot sklearn
    axes[1, idx].scatter(X_compare[:, 0], X_compare[:, 1], c=sklearn_labels, 
                         cmap='viridis', s=30, alpha=0.7)
    axes[1, idx].set_title(f'sklearn - {linkage.capitalize()}')

axes[0, 0].set_ylabel('Our Implementation')
axes[1, 0].set_ylabel('sklearn')

plt.suptitle('Cluster Assignments: Our Implementation vs sklearn', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Timing comparison with different dataset sizes
print("\nTiming Comparison (seconds)")
print("=" * 50)
print("Note: Our implementation is for educational purposes.")
print("sklearn is highly optimized for production use.")
print()

sizes = [50, 100, 150, 200]
timing_results = {'our': [], 'sklearn': []}

for n_samples in sizes:
    X_timing, _ = make_blobs(n_samples=n_samples, n_features=2, centers=3, random_state=42)
    X_timing = StandardScaler().fit_transform(X_timing)
    
    # Our implementation
    start = time.time()
    model = AgglomerativeClustering(n_clusters=3, linkage='ward')
    model.fit_predict(X_timing)
    our_time = time.time() - start
    timing_results['our'].append(our_time)
    
    # sklearn
    start = time.time()
    sklearn_model = SklearnAgglomerative(n_clusters=3, linkage='ward')
    sklearn_model.fit_predict(X_timing)
    sklearn_time = time.time() - start
    timing_results['sklearn'].append(sklearn_time)

# Print timing table
print(f"{'n_samples':<12} {'Our (s)':>12} {'sklearn (s)':>12} {'Ratio':>12}")
print("-" * 50)
for i, n in enumerate(sizes):
    ratio = timing_results['our'][i] / timing_results['sklearn'][i]
    print(f"{n:<12} {timing_results['our'][i]:>12.4f} {timing_results['sklearn'][i]:>12.4f} {ratio:>12.1f}x")

# Plot timing comparison
plt.figure(figsize=(10, 5))
plt.plot(sizes, timing_results['our'], 'bo-', label='Our Implementation', linewidth=2, markersize=8)
plt.plot(sizes, timing_results['sklearn'], 'ro-', label='sklearn', linewidth=2, markersize=8)
plt.xlabel('Number of Samples')
plt.ylabel('Time (seconds)')
plt.title('Timing Comparison: Our Implementation vs sklearn')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Summary & Key Takeaways

### What We Learned:

1. **Hierarchical Clustering Fundamentals**
   - Agglomerative (bottom-up) vs Divisive (top-down) approaches
   - Different linkage methods and their characteristics
   - Dendrogram interpretation and cutting

2. **Implementation Details**
   - Distance matrix computation: O(n^2) space
   - Cluster merging with different linkage criteria
   - Label assignment by cutting the dendrogram

3. **Evaluation Metrics**
   - Silhouette score for cluster quality
   - Cophenetic correlation for dendrogram faithfulness
   - Adjusted Rand Index for comparison with ground truth

4. **Practical Considerations**
   - O(n^2) memory limits dataset size
   - Linkage choice affects cluster shapes
   - Single linkage susceptible to chaining

### Linkage Method Summary:

| Method | Pros | Cons |
|--------|------|------|
| Single | Finds elongated clusters | Chaining effect |
| Complete | Compact clusters | Sensitive to outliers |
| Average | Balanced approach | Computationally heavier |
| Ward | Equal-sized clusters | Assumes spherical shape |

### Key Insights:

- Hierarchical clustering is powerful for exploratory analysis
- Dendrograms provide interpretable cluster relationships
- Linkage choice significantly impacts results
- Not suitable for large datasets due to O(n^2) complexity
- sklearn provides highly optimized production implementation

### Next Steps:

- Experiment with different distance metrics (cosine, manhattan)
- Try BIRCH or HDBSCAN for larger datasets
- Combine with dimensionality reduction (PCA, t-SNE)
- Apply to real-world hierarchical data (taxonomies, documents)